Preparations:
- load prediction.csv
    - drop the lines with the non-used ifs
    - if there is more then one if, take mean

- add column "sample" to master table
- join them
- drop unnecessary columns (matadata, paths, ranks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import ast
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score, precision_recall_curve, confusion_matrix

# Configuration
CSV_PATH =  "/home/bri/pymol/data/predictions/preds_mcmahon_ad3.csv" 
MASTER_PATH =  "/home/bri/pymol/data/mcmahon/table/master_table_mcmahon.csv" 
YAML_PATH = "/home/bri/pymol/EDA/static/metric_directions.yaml"
OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/adiponectin3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


KEEP_INTERFACES = {"A,C", "B,C"}   # as option if there is more than one, default A,B



# Load data
preds_df = pd.read_csv(CSV_PATH)
preds_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 92 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   sample                                  105 non-null    str    
 1   binding                                 105 non-null    bool   
 2   interface                               105 non-null    str    
 3   af3_pae                                 105 non-null    float64
 4   af3_plddt                               105 non-null    float64
 5   af3_interface_dG                        105 non-null    float64
 6   af3_interface_dG_SASA_ratio             105 non-null    float64
 7   af3_ipae                                105 non-null    float64
 8   af3_iplddt                              105 non-null    float64
 9   af3_ipsae                               105 non-null    float64
 10  af3_pdockq                              105 non-null    float64
 11  af3_

#### snir

In [2]:
df_if = preds_df[preds_df["interface"].str.strip().isin(KEEP_INTERFACES)].copy()
print(f"Loaded {len(df_if)} rows. Interfaces: {df_if['interface'].unique()}")

df_if.head(10)


Loaded 70 rows. Interfaces: <StringArray>
['A,C', 'B,C']
Length: 2, dtype: str


,sample,binding,interface,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,...,boltz_template_plddt,boltz_template_iplddt,boltz_template_pde,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_model_path,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis
1,5ijk_acx_cut_A_I114A,False,"A,C",3.581104,91.142842,52.43,6.22,4.484615,65.463607,0.023617,...,0.971302,0.964612,0.263253,0.334753,0.892070,/scicore/home/schwede/barta0000/scoring_pipeli...,0.798145,0.0303,0.7167,0.7665
2,5ijk_acx_cut_A_I114A,False,"B,C",3.581104,91.142842,83.71,14.76,1.594444,70.043835,0.024757,...,0.971302,0.964612,0.263253,0.334753,0.888323,/scicore/home/schwede/barta0000/scoring_pipeli...,0.774026,0.1712,0.8551,0.7862
4,5ifj_abc_cut_B_V38A,False,"A,C",2.714935,95.220947,17.70,2.18,2.678125,92.334169,0.758807,...,0.983234,0.981698,0.262260,0.333394,0.904294,/scicore/home/schwede/barta0000/scoring_pipeli...,0.831587,0.2632,0.9405,0.8296
5,5ifj_abc_cut_B_V38A,False,"B,C",2.714935,95.220947,13.18,1.82,1.830000,94.597707,0.758785,...,0.983234,0.981698,0.262260,0.333394,0.920807,/scicore/home/schwede/barta0000/scoring_pipeli...,0.848706,0.2098,0.9402,0.8144
7,5ifj_abc_cut_B_Y37A,False,"A,C",3.004751,94.723352,14.88,1.87,2.880556,91.827094,0.665555,...,0.979142,0.971864,0.260346,0.321740,0.921565,/scicore/home/schwede/barta0000/scoring_pipeli...,0.865535,0.2102,0.9206,0.8200
8,5ifj_abc_cut_B_Y37A,False,"B,C",3.004751,94.723352,6.63,0.89,1.716667,92.413659,0.615602,...,0.979142,0.971864,0.260346,0.321740,0.887115,/scicore/home/schwede/barta0000/scoring_pipeli...,0.782346,0.1767,0.8625,0.8042
10,5ig7_abc_cut,False,"A,C",2.572969,95.599717,10.97,1.38,2.321875,94.630252,0.780104,...,0.987334,0.987299,0.258769,0.284754,0.948463,/scicore/home/schwede/barta0000/scoring_pipeli...,0.921477,0.2352,0.9699,0.8588
11,5ig7_abc_cut,False,"B,C",2.572969,95.599717,5.13,0.67,1.720000,95.189670,0.790385,...,0.987334,0.987299,0.258769,0.284754,0.956973,/scicore/home/schwede/barta0000/scoring_pipeli...,0.928806,0.2332,0.9688,0.8586
13,5ifj_abc_cut_A_D111A,False,"A,C",2.815847,95.282854,15.90,2.09,2.450000,94.521058,0.754481,...,0.979963,0.975488,0.256625,0.289870,0.938492,/scicore/home/schwede/barta0000/scoring_pipeli...,0.903545,0.2228,0.9224,0.8619
14,5ifj_abc_cut_A_D111A,False,"B,C",2.815847,95.282854,10.55,1.40,1.750000,94.940492,0.776109,...,0.979963,0.975488,0.256625,0.289870,0.949190,/scicore/home/schwede/barta0000/scoring_pipeli...,0.913493,0.2168,0.9356,0.8502


#### all

In [3]:
num_cols = df_if.select_dtypes(include=['number']).columns
other_cols = [c for c in df_if.columns if c not in num_cols and c != 'sample']

agg_dict = {col: 'mean' for col in num_cols}
agg_dict.update({col: 'first' for col in other_cols})

df_1if = df_if.groupby('sample').agg(agg_dict).reset_index()

df_1if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,binding,interface,chai_unconstrained_model_path,chai_constrained_model_path,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm,boltz_free_model_path,boltz_template_model_path
0,5ifj_abc_cut,2.593883,95.404666,14.690,1.855,1.993750,95.036725,0.788184,0.20980,0.88085,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9454320669174194, 0.9492036700248718, 0.17...","[[[0.9454320669174194, 0.8899917602539062, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
1,5ifj_abc_cut_A_D111A,2.815847,95.282854,13.225,1.745,2.100000,94.730775,0.765295,0.18660,0.88195,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.9454193711280823, 0.9488046169281006, 0.17...","[[[0.9454193711280823, 0.8907997608184814, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
2,5ifj_abc_cut_A_K110A,2.790728,94.871984,19.975,2.845,2.275417,92.998114,0.769552,0.18030,0.85065,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9449247717857361, 0.9489190578460693, 0.17...","[[[0.9449247717857361, 0.889130175113678, 0.07...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
3,5ifj_abc_cut_A_S64A,2.539446,95.613237,14.595,1.810,2.072188,94.813927,0.778962,0.19900,0.88450,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.945385217666626, 0.948758602142334, 0.1724...","[[[0.945385217666626, 0.8885297775268555, 0.07...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
4,5ifj_abc_cut_A_W52A,3.005329,94.814333,46.435,5.940,2.303125,91.482646,0.695928,0.19125,0.78185,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9452540874481201, 0.9489129781723022, 0.17...","[[[0.9452540874481201, 0.8904703855514526, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
5,5ifj_abc_cut_A_Y66A,2.701376,95.337838,17.485,2.305,2.249649,93.429048,0.763715,0.19335,0.84245,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9473956227302551, 0.9488828778266907, 0.16...","[[[0.9473956227302551, 0.8850024938583374, 0.0...","[[0.9453608989715576, 0.9489309191703796, 0.17...","[[[0.9453608989715576, 0.8903481960296631, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
6,5ifj_abc_cut_B_L116A,2.806607,95.032996,7.610,0.985,2.205952,93.484476,0.747441,0.17235,0.82990,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9485109448432922, 0.9499985575675964, 0.17...","[[[0.9485109448432922, 0.8920019865036011, 0.0...","[[0.9452025294303894, 0.9490475058555603, 0.17...","[[[0.9452025294303894, 0.8905109763145447, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/

In [4]:
df_2if = df_1if.drop(columns=["binding", "interface","chai_unconstrained_model_path", "chai_constrained_model_path", "boltz_free_model_path", "boltz_template_model_path"])
df_2if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm
0,5ifj_abc_cut,2.593883,95.404666,14.690,1.855,1.993750,95.036725,0.788184,0.20980,0.88085,...,0.278017,0.960036,0.937455,0.23935,0.96685,0.87115,"[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9454320669174194, 0.9492036700248718, 0.17...","[[[0.9454320669174194, 0.8899917602539062, 0.0..."
1,5ifj_abc_cut_A_D111A,2.815847,95.282854,13.225,1.745,2.100000,94.730775,0.765295,0.18660,0.88195,...,0.289870,0.943841,0.908519,0.21980,0.92900,0.85605,"[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.9454193711280823, 0.9488046169281006, 0.17...","[[[0.9454193711280823, 0.8907997608184814, 0.0..."
2,5ifj_abc_cut_A_K110A,2.790728,94.871984,19.975,2.845,2.275417,92.998114,0.769552,0.18030,0.85065,...,0.275365,0.961354,0.939275,0.23650,0.96220,0.87810,"[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9449247717857361, 0.9489190578460693, 0.17...","[[[0.9449247717857361, 0.889130175113678, 0.07..."
3,5ifj_abc_cut_A_S64A,2.539446,95.613237,14.595,1.810,2.072188,94.813927,0.778962,0.19900,0.88450,...,0.295100,0.938922,0.898855,0.22850,0.95750,0.84545,"[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.945385217666626, 0.948758602142334, 0.1724...","[[[0.945385217666626, 0.8885297775268555, 0.07..."
4,5ifj_abc_cut_A_W52A,3.005329,94.814333,46.435,5.940,2.303125,91.482646,0.695928,0.19125,0.78185,...,0.343705,0.913520,0.845670,0.21780,0.94175,0.81200,"[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9452540874481201, 0.9489129781723022, 0.17...","[[[0.9452540874481201, 0.8904703855514526, 0.0..."
5,5ifj_abc_cut_A_Y66A,2.701376,95.337838,17.485,2.305,2.249649,93.429048,0.763715,0.19335,0.84245,...,0.305202,0.935591,0.892648,0.20680,0.95410,0.83435,"[[0.9473956227302551, 0.9488828778266907, 0.16...","[[[0.9473956227302551, 0.8850024938583374, 0.0...","[[0.9453608989715576, 0.9489309191703796, 0.17...","[[[0.9453608989715576, 0.8903481960296631, 0.0..."
6,5ifj_abc_cut_B_L116A,2.806607,95.032996,7.610,0.985,2.205952,93.484476,0.747441,0.17235,0.82990,...,0.290632,0.941965,0.905323,0.21785,0.94930,0.85275,"[[0.9485109448432922, 0.9499985575675964, 0.17...","[[[0.9485109448432922, 0.8920019865036011, 0.0...","[[0.9452025294303894, 0.9490475058555603, 0.17...","[[[0.9452025294303894, 0.8905109763145447, 0.0..."
7,5ifj_abc_cut_B_V38A,2.714935,95.220947,15.440,2.000,2.254062,93.465938,0.758796,0.19055,0.85940,...,0.333394,0.912550,0.840146,0.23650,0.94035,0.82200,"[[0.9468277096748352, 0.9472519159317017, 0.10...","[[[0.9468277096748352, 0.8879522085189819, 0.0...","[[0.945318341255188, 0.9495519995689392, 0.177...","[[[0.945318341255188, 0.890434980392456, 0.072..."
8,5ifj_abc_cut_B_W107A,2.878966,95.089607,17.035,2.335,2.471875,93.380576,0.623665,0.16430,0.79175,...,0.315265,0.916625,0.851439,0.21350,0.93755,0.83195,"[[0.9425944089889526, 0.9466992616653442, 0.10...","[[[0.9425944089889526, 0.8834366798400879, 0.0...","[[0.9448549747467041, 0.9490682482719421, 0.17...","[[[0.9448549747467041, 0.8899660706520081, 0.0..."
9,5ifj_abc_cut_B_Y37A,3.004751,94.723352,10.755,1.380,2.298611,92.120377,0.640578,0.17745,0.77900,...,0.321740,0.904340,0.823940,0.19345,0.89155,0.81210,"[[0.940564751625061, 0.9448128938674927, 0.095...","[[[0.940564751625061, 0.8819113969802856, 0.04...","[[0.9450069665908813, 0.94929438829422, 0.1781...","[[[0.9450069665908813, 0.889803946018

## master df

##### snir


In [5]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df

,source,type,sample,fab,interface,vh/vl,target,target_residues,target_sequence,binder,KD[M],notes
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,1,0.000001,Fab 1E01 + KD[M] approximated from Snir et. al...
...,...,...,...,...,...,...,...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLASSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN


In [6]:
master_df2 = master_df.drop(columns=["interface", "fab", "vh/vl", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,sample,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1,0.000010
1,snir_ab,original,5ifj_abc_cut,1,0.000010
2,snir_ab,original,5ig7_abc_cut,1,0.000100
3,snir_ab,original,5ig7_abc_cut,1,0.000100
4,snir_ab,original,5ijk_acx_cut,1,0.000001
...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,0,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,0,NaN


In [7]:
master_df3 = master_df2.drop_duplicates(keep="last")
master_df3

,source,type,sample,binder,KD[M]
1,snir_ab,original,5ifj_abc_cut,1,0.000010
3,snir_ab,original,5ig7_abc_cut,1,0.000100
5,snir_ab,original,5ijk_acx_cut,1,0.000001
7,snir_ab,alanine_scan,5ifj_abc_cut_A_D111A,0,NaN
9,snir_ab,alanine_scan,5ifj_abc_cut_A_K110A,0,NaN
11,snir_ab,alanine_scan,5ifj_abc_cut_A_S64A,0,NaN
13,snir_ab,alanine_scan,5ifj_abc_cut_A_W52A,0,NaN
15,snir_ab,alanine_scan,5ifj_abc_cut_A_Y66A,0,NaN
17,snir_ab,alanine_scan,5ifj_abc_cut_all_A,0,NaN
19,snir_ab,alanine_scan,5ifj_abc_cut_B_L116A,0,NaN


##### mcmahon / germinal

In [ ]:

master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df.head()

,source,type,sample,fab,interface,vh/vl,target,target_residues,target_sequence,binder,KD[M],notes
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,1,0.000001,Fab 1E01 + KD[M] approximated from Snir et. al...


In [5]:
AMBIGUOUS_TARGET= "HSA"   # test

def create_sample_id(row):
    vhh = str(row['vhh'])
    target = str(row['target'])
    residues = str(row['target_residues'])
    
    # Check if target is (ambigous target, meaning the same target has different version --> target residues are needed to be able to identify it, the only one for now is HSA) to apply the specific naming convention
    if target == AMBIGUOUS_TARGET:
        return f"{vhh}_{target}_{residues}"
    else:
        return f"{vhh}_{target}"

# 1. Create the 'sample' column in the master table
master_df['sample'] = master_df.apply(create_sample_id, axis=1)
master_df



,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],sample
0,germinal,original,nb_s413501_mpnn_3,QVQLVESGGGLVQPGGSLRLSCAASGSRAIHLKYVGLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s413501_mpnn_3_PDL1
1,germinal,original,nb_s80903_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASGSLYISNCKTHLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s80903_mpnn_4_PDL1
2,germinal,original,nb_s185784_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASRWFSNEFHNICLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s185784_mpnn_4_PDL1
3,germinal,original,nb_s465730_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASSANFSLSGFHHLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s465730_mpnn_4_PDL1
4,germinal,original,nb_s892360_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASSRFPWEFIRMWLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s892360_mpnn_4_PDL1
...,...,...,...,...,...,...,...,...,...,...
475,germinal,target_shuffle,nb_s276607_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASGDDVDKWFLDWLGWFRQAPGQ...,IL20,25-176,LKTLNLGSCVIATNLQEIRNGFSEIRGSVQAKDGNIDIRILRRTES...,0,NaN,nb_s276607_mpnn_4_IL20
476,germinal,target_shuffle,nb_s442197_mpnn_3,QVQLVESGGGLVQPGGSLRLSCAASFDLSRCYMISYLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s442197_mpnn_3_PDL1
477,germinal,target_shuffle,nb_s776407_mpnn_2,QVQLVESGGGLVQPGGSLRLSCAASDFFTSKYHWWYLGWFRQAPGQ...,IL3,31-152,SWVNCSNMIDEIITHLKQPPLPLLDFNNLNGEDQDILMENNLRRPN...,0,NaN,nb_s776407_mpnn_2_IL3
478,germinal,target_shuffle,nb_s206651_mpnn_1,QVQLVESGGGLVQPGGSLRLSCAASGMTAAEFKERELGWFRQAPGQ...,IL3,31-152,SWVNCSNMIDEIITHLKQPPLPLLDFNNLNGEDQDILMENNLRRPN...,0,NaN,nb_s206651_mpnn_1_IL3


In [6]:
master_df2 = master_df.drop(columns=["vhh", "vhh_sequence", "target", "target_residues", "target_sequence"])
master_df2

,source,type,binder,KD[M],sample
0,germinal,original,0,NaN,nb_s413501_mpnn_3_PDL1
1,germinal,original,0,NaN,nb_s80903_mpnn_4_PDL1
2,germinal,original,0,NaN,nb_s185784_mpnn_4_PDL1
3,germinal,original,0,NaN,nb_s465730_mpnn_4_PDL1
4,germinal,original,0,NaN,nb_s892360_mpnn_4_PDL1
...,...,...,...,...,...
475,germinal,target_shuffle,0,NaN,nb_s276607_mpnn_4_IL20
476,germinal,target_shuffle,0,NaN,nb_s442197_mpnn_3_PDL1
477,germinal,target_shuffle,0,NaN,nb_s776407_mpnn_2_IL3
478,germinal,target_shuffle,0,NaN,nb_s206651_mpnn_1_IL3


### merge

In [8]:
scores_df = pd.merge(master_df3, df_2if, on=['sample'], how='inner')
#df = pd.merge(master_df2, preds_df, on=['sample'], how='inner')
# Display the first few rows to verify
print(scores_df[['sample', 'af3_plddt']])

# Save the merged result
scores_df.to_csv(OUTPUT_DIR /'merged_snir.csv', index=False)

                  sample  af3_plddt
0           5ifj_abc_cut  95.404666
1           5ig7_abc_cut  95.599717
2           5ijk_acx_cut  92.439163
3   5ifj_abc_cut_A_D111A  95.282854
4   5ifj_abc_cut_A_K110A  94.871984
5    5ifj_abc_cut_A_S64A  95.613237
6    5ifj_abc_cut_A_W52A  94.814333
7    5ifj_abc_cut_A_Y66A  95.337838
8     5ifj_abc_cut_all_A  93.235998
9   5ifj_abc_cut_B_L116A  95.032996
10   5ifj_abc_cut_B_V38A  95.220947
11  5ifj_abc_cut_B_W107A  95.089607
12   5ifj_abc_cut_B_Y37A  94.723352
13  5ig7_abc_cut_A_K110A  94.950463
14   5ig7_abc_cut_A_W52A  95.457781
15   5ig7_abc_cut_A_Y66A  95.767986
16    5ig7_abc_cut_all_A  92.210506
17  5ig7_abc_cut_B_G114A  95.869087
18  5ig7_abc_cut_B_L116A  95.298420
19   5ig7_abc_cut_B_V38A  95.793462
20  5ig7_abc_cut_B_W107A  92.830579
21   5ig7_abc_cut_B_Y37A  95.631059
22  5ijk_acx_cut_A_I112A  94.094014
23  5ijk_acx_cut_A_I114A  91.142842
24   5ijk_acx_cut_A_K57A  92.231214
25   5ijk_acx_cut_A_R55A  91.770396
26  5ijk_acx_cut_A_T113A  97